# QNE+GBFC Hybrid on small-graph — the leaderboard shot (Kaggle GPU)

The winner reaches the top via WIDE per-threshold neuroevolution (~100k gens).
GBFC alone plateaus because it stays in our basin. This **hybrid** does both:
per-threshold neuroevolution (wide exploration, seeded partly from our −1,828,994
so it never regresses) **+** periodic GBFC GBDT-boost of the worst band. small is
fast, so ~30 GPU-hours ≈ 100k+ generations — in range of the winner's compute.

Checkpoint-safe (submission rewritten periodically). **Settings → GPU T4 ×2,
Internet ON. Add Data → Upload `torso_project.zip`.** Then Run All.

In [ ]:
import os, glob, zipfile
cands = glob.glob('/kaggle/input/**/torso_project.zip', recursive=True) + glob.glob('torso_project.zip')
assert cands, "Upload torso_project.zip via Add Data."
os.makedirs('/kaggle/working/run', exist_ok=True)
zipfile.ZipFile(cands[0]).extractall('/kaggle/working/run')
ROOT = os.path.dirname(glob.glob('/kaggle/working/run/**/tools/qnegbfc.py', recursive=True)[0]).rsplit('/tools',1)[0]
os.chdir(ROOT); print('cwd:', os.getcwd())

In [ ]:
!pip -q install lightgbm numba 2>/dev/null
from numba import cuda; print('CUDA:', cuda.is_available())   # must be True

## Run the hybrid (~10 h; rerun across sessions for ~30 h)
`--budget 36000` ≈ 10 h, fits one Kaggle session. It prints the **official** score
periodically and approaches −1,829,919. Seeds from −1,828,994 and explores wider.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
!PYTHONWARNINGS=ignore python3 tools/qnegbfc.py --problem small-graph --batch 1024 --budget 36000 --inject-every 40 --algo qnegbfc

## Save + (optional) the controlled ablation
Save the result; download from the Output tab and re-score on your machine.

In [ ]:
import shutil; shutil.copy('submissions/small-graph/qnegbfc.json', '/kaggle/working/qnegbfc_small.json')
print('saved /kaggle/working/qnegbfc_small.json')

**Novelty ablation** — run this in a *separate* session to isolate GBDT's
contribution to the SOTA method (same engine, GBFC injection OFF):
```
python3 tools/qnegbfc.py --problem small-graph --batch 1024 --budget 36000 --no-gbdt --algo qne_wide
```
If `qnegbfc` (GBDT on) beats `qne_wide` (GBDT off), that gap is your novel
contribution — *GBDT boosting exceeds the state-of-the-art neuroevolution.*

**Re-score on your Mac** (pools with your banked best, so it never regresses):
```
cp ~/Downloads/qnegbfc_small.json submissions/small-graph/
python3 tools/portfolio.py --problems small-graph
python3 tools/refine_thresholds.py --problems small-graph
```
Watch the official line for −1,829,919 or below = #1 on small.